# Fatias da imagem, círculos e extensão da aorta

Analisa a cobertura axial dos círculos da aorta e compara essa trajetória com as fatias efetivamente ocupadas pela máscara final. Valores positivos na variação indicam expansão axial da segmentação; valores negativos indicam recuo.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

# Localiza a raiz antes dos imports internos a partir de qualquer subpasta de EDA.
current = Path.cwd().resolve()
REPO_ROOT = next(
    path
    for path in [current, *current.parents]
    if (path / "src").exists() and (path / "output").exists()
)
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils.experiments import (  # noqa: E402
    get_aorta_visual_review,
    load_aorta_review_cohort,
    load_aorta_visual_reviews,
)
from utils.project.runtime.notebook_env import configure_notebook_environment  # noqa: E402

REPO_ROOT = configure_notebook_environment(chdir_to_src=False)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

## 1. Configuração

As duas coortes usam o pipeline normal com limite inferior de -300 HU e limite superior P99.9. Os rótulos de aorta boa/ruim vêm da inspeção visual dos HTMLs 3D.

In [ ]:
REVIEW_CONFIG_PATH = REPO_ROOT / "config/aorta_visual_reviews.json"
REVIEW_CATALOG = load_aorta_visual_reviews(REVIEW_CONFIG_PATH)
TRAIN_REVIEW = get_aorta_visual_review(REVIEW_CATALOG, "normal", "train")
VAL_REVIEW = get_aorta_visual_review(REVIEW_CATALOG, "normal", "val")

print(
    f"Treino: {len(TRAIN_REVIEW['aorta_good_ids'])} boas e "
    f"{len(TRAIN_REVIEW['aorta_bad_ids'])} ruins"
)
print(
    f"Validação: {len(VAL_REVIEW['aorta_good_ids'])} boas e "
    f"{len(VAL_REVIEW['aorta_bad_ids'])} ruins"
)

## 2. Carregamento e métricas derivadas

`aorta_circle_count` mede as fatias representadas pela trajetória final de círculos. `aorta_segmented_slice_count` conta as fatias que possuem ao menos um voxel na máscara final da aorta.

In [ ]:
REQUIRED_COLUMNS = {
    "IMG_ID",
    "image_slice_count",
    "aorta_circle_count",
    "aorta_detected_circle_count",
    "aorta_interpolated_circle_count",
    "aorta_circle_first_slice",
    "aorta_circle_last_slice",
    "aorta_circle_radius_mean_mm",
    "aorta_circle_radius_std_mm",
    "aorta_segmented_slice_count",
    "ostia_detection_status",
    "artery_dice",
}

train_df = load_aorta_review_cohort(
    REPO_ROOT,
    TRAIN_REVIEW,
    "train",
    cohort_name="treino",
    required_columns=REQUIRED_COLUMNS,
    use_reviewed_ostia_labels=True,
)
val_df = load_aorta_review_cohort(
    REPO_ROOT,
    VAL_REVIEW,
    "val",
    cohort_name="validação",
    required_columns=REQUIRED_COLUMNS,
    use_reviewed_ostia_labels=True,
)
combined_df = pd.concat([train_df, val_df], ignore_index=True)

print(f"Treino carregado: {len(train_df)} imagens")
print(f"Validação carregada: {len(val_df)} imagens")

## 3. Visão geral das coortes

In [ ]:
def summarize_cohort(cohort_df, cohort_name):
    # Resume círculos, máscara final, raio e desempenho da coorte.
    return {
        "coorte": cohort_name,
        "imagens": len(cohort_df),
        "fatias_imagem_media": cohort_df["image_slice_count"].mean(),
        "fatias_com_circulo_media": cohort_df["aorta_circle_count"].mean(),
        "fatias_segmentadas_media": cohort_df["aorta_segmented_slice_count"].mean(),
        "variacao_fatias_media": cohort_df["segmented_minus_circle_slices"].mean(),
        "variacao_percentual_media": (
            cohort_df["segmented_vs_circle_change_fraction"].mean() * 100
        ),
        "cobertura_circulos_media_percentual": (
            cohort_df["circle_slice_fraction"].mean() * 100
        ),
        "cobertura_mascara_media_percentual": (
            cohort_df["segmented_slice_fraction"].mean() * 100
        ),
        "raio_medio_mm": cohort_df["aorta_circle_radius_mean_mm"].mean(),
        "sucesso_ostios_percentual": cohort_df["ostia_success"].mean() * 100,
        "dice_arterial_medio": cohort_df["artery_dice"].mean(),
    }


overview_df = pd.DataFrame(
    [
        summarize_cohort(train_df, "treino"),
        summarize_cohort(val_df, "validação"),
    ]
)
display(overview_df.round(3))

## 4. Relação entre círculos e máscara final

A variação é calculada por

\[
\Delta N = N_{\mathrm{segmentadas}} - N_{\mathrm{círculos}}.
\]

- `ΔN > 0`: a máscara avançou além da quantidade de fatias representadas pelos círculos;
- `ΔN = 0`: as extensões são iguais;
- `ΔN < 0`: houve recuo axial após level set e pós-processamento.

A variação percentual usa a quantidade de fatias com círculos como referência.

In [ ]:
extent_summary_df = (
    combined_df.assign(
        direcao_extensao=combined_df["segmented_minus_circle_slices"].map(
            lambda value: (
                "expansão" if value > 0 else ("recuo" if value < 0 else "igual")
            )
        )
    )
    .groupby(["coorte", "visual_aorta_quality"], observed=True)
    .agg(
        imagens=("IMG_ID", "size"),
        fatias_com_circulo_media=("aorta_circle_count", "mean"),
        fatias_segmentadas_media=("aorta_segmented_slice_count", "mean"),
        variacao_fatias_media=("segmented_minus_circle_slices", "mean"),
        variacao_fatias_mediana=("segmented_minus_circle_slices", "median"),
        variacao_percentual_media=("segmented_vs_circle_change_fraction", "mean"),
        exames_com_expansao=(
            "direcao_extensao",
            lambda values: values.eq("expansão").sum(),
        ),
        exames_com_recuo=("direcao_extensao", lambda values: values.eq("recuo").sum()),
        exames_sem_variacao=(
            "direcao_extensao",
            lambda values: values.eq("igual").sum(),
        ),
        sucesso_ostios_percentual=("ostia_success", "mean"),
    )
    .reset_index()
)
extent_summary_df["variacao_percentual_media"] *= 100
extent_summary_df["sucesso_ostios_percentual"] *= 100
display(extent_summary_df.round(3))

## 5. Círculos por qualidade visual da aorta

Esta tabela mantém as informações de cobertura, raio e posição que ajudam a distinguir trajetórias boas e ruins, sem repetir as mesmas relações em vários gráficos.

In [ ]:
quality_summary_df = (
    combined_df.groupby(["coorte", "visual_aorta_quality"], observed=True)
    .agg(
        imagens=("IMG_ID", "size"),
        fatias_imagem_media=("image_slice_count", "mean"),
        cobertura_circulos_media=("circle_slice_fraction", "mean"),
        cobertura_mascara_media=("segmented_slice_fraction", "mean"),
        raio_medio_mm=("aorta_circle_radius_mean_mm", "mean"),
        variacao_media_raio_mm=("aorta_circle_radius_std_mm", "mean"),
        primeira_posicao_media=("circle_first_position", "mean"),
        centro_posicao_media=("circle_center_position", "mean"),
        ultima_posicao_media=("circle_last_position", "mean"),
        sucesso_ostios=("ostia_success", "mean"),
        dice_medio=("artery_dice", "mean"),
    )
    .reset_index()
)
percentage_columns = [
    "cobertura_circulos_media",
    "cobertura_mascara_media",
    "primeira_posicao_media",
    "centro_posicao_media",
    "ultima_posicao_media",
    "sucesso_ostios",
]
quality_summary_df[percentage_columns] *= 100
display(quality_summary_df.round(3))

## 6. Valores por exame

A tabela começa pelas aortas ruins e mostra diretamente quanto a máscara avançou ou recuou em relação à trajetória de círculos.

In [ ]:
case_table_df = combined_df.assign(
    cobertura_circulos_percentual=combined_df["circle_slice_fraction"] * 100,
    cobertura_mascara_percentual=combined_df["segmented_slice_fraction"] * 100,
    variacao_percentual=(combined_df["segmented_vs_circle_change_fraction"] * 100),
).sort_values(
    ["coorte", "visual_aorta_quality", "segmented_minus_circle_slices"],
    ascending=[True, False, True],
)[
    [
        "coorte",
        "IMG_ID",
        "visual_aorta_quality",
        "ostia_outcome",
        "image_slice_count",
        "aorta_circle_count",
        "aorta_segmented_slice_count",
        "segmented_minus_circle_slices",
        "variacao_percentual",
        "cobertura_circulos_percentual",
        "cobertura_mascara_percentual",
        "aorta_detected_circle_count",
        "aorta_interpolated_circle_count",
        "aorta_circle_radius_mean_mm",
        "aorta_circle_radius_std_mm",
        "artery_dice",
    ]
]
display(case_table_df.round(3))